# Exercise 1: Streaming Text Generation with Amazon Bedrock

This notebook demonstrates how to use Amazon Bedrock's **Nova Micro** model to generate and transform text using the `invoke_model_with_response_stream` API for real-time streaming responses.

## Overview

This exercise shows:
- How to use the `invoke_model_with_response_stream` API for streaming text generation
- How to process model responses incrementally as they are generated
- How to handle streaming JSON chunks
- How to display text in real-time as it's generated

## Prerequisites

- AWS account configured with appropriate credentials
- Amazon Bedrock access enabled for Nova Micro model
- AWS CLI configured with credentials

## Model Used

- **Model ID**: `amazon.nova-micro-v1:0`
- **Model Type**: Text generation and editing
- **API**: `invoke_model_with_response_stream` (streaming processing)

## Key Benefits of Streaming

- **Real-time feedback**: See responses as they're generated
- **Better UX**: Users see progress immediately
- **Lower latency**: Process output incrementally
- **Efficiency**: Can stop processing early if needed

## Use Case

This example asks the model to describe different types of dances, with the response streaming in real-time.



## Step 1: Import Required Libraries

Import the necessary Python libraries:
- `boto3`: AWS SDK for Python
- `json`: For parsing streaming JSON chunks from the API response


## Step 2: Stream Text Generation

This cell:
1. Creates a Bedrock Runtime client for the us-east-1 region
2. Sets the model ID to Nova Micro
3. Prepares a payload with:
   - **Messages**: User prompt asking about types of dances
   - **Content Type**: application/json
   - **Accept**: application/json
4. Calls `invoke_model_with_response_stream` to start streaming
5. Processes the response stream:
   - Iterates through stream events
   - Extracts chunk bytes and decodes to UTF-8
   - Parses JSON chunks
   - Extracts text deltas from `contentBlockDelta`
   - Prints text incrementally as it's generated
   - Accumulates full text for later use
6. Handles JSON decode errors gracefully

**How Streaming Works**:
- The API returns a stream of events
- Each event contains a chunk with JSON data
- The `contentBlockDelta` contains incremental text updates
- Text is printed character-by-character or word-by-word as it arrives
- The full response is accumulated in `full_text` variable

**Note**: This is an asynchronous streaming API - responses arrive incrementally, allowing for real-time display of generated content.

**Expected Output**: You'll see the response text appear gradually in the output, describing various types of dances from around the world.


In [1]:
import boto3
import json

In [2]:
bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')
model_id = 'amazon.nova-micro-v1:0'

payload = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "text": "Tell me what type of dances people do."
                }
            ]
        }
    ]
}

payload_json = json.dumps(payload)

response = bedrock.invoke_model_with_response_stream(
    modelId=model_id,
    body=payload_json,
    contentType='application/json',
    accept='application/json'
)

stream = response['body']
full_text = ""

print("Receiving response stream:")
for event in stream:
    chunk = event['chunk']
    chunk_str = chunk.get('bytes', b'').decode('utf-8')

    try:
        json_chunk = json.loads(chunk_str)

        if "contentBlockDelta" in json_chunk:
            delta = json_chunk["contentBlockDelta"]["delta"]
            text = delta.get("text", "")
            full_text += text
            print(text, end='') 

    except json.JSONDecodeError:
        continue

Receiving response stream:
There are numerous types of dances that people perform around the world, each with its own unique style, history, and cultural significance. Here are some of the most popular and widely recognized types of dances:

### Traditional and Folk Dances
1. **Ballet** - A classical dance style that originated in Italy and France, characterized by its grace, precision, and use of music.
2. **Hip-Hop** - A street dance style that includes various forms like breaking, locking, and popping, often accompanied by rap music.
3. **Salsa** - A lively dance from Latin America, particularly popular in Cuba and Puerto Rico, characterized by its intricate footwork and partner work.
4. **Tango** - An elegant and passionate dance that originated in the late 19th century in Argentina and Uruguay.
5. **Flamenco** - A traditional Spanish dance from the Andalusian region, known for its passionate music, intricate footwork, and expressive hand and body movements.
6. **Bhangra** - A trad